# Local RAG with Gemma 4 + Ollama

This notebook demonstrates how to build a **fully local** RAG (Retrieval-Augmented Generation) pipeline using:
- **Ollama** for the LLM (runs locally, no API keys needed)
- **HuggingFace Embeddings** for document embedding (sentence-transformers)
- **LlamaIndex** for orchestrating the RAG pipeline

## RAG Pipeline Steps

1. **Load** - Read the PDF document
2. **Chunk** - Split into manageable pieces
3. **Embed** - Convert chunks to vector representations
4. **Index** - Store vectors for efficient retrieval
5. **Query** - Retrieve relevant chunks and generate answers

## Prerequisites

1. **Install Ollama**: Download from [ollama.ai](https://ollama.ai)
2. **Pull a model**: Run `ollama pull gemma4` in your terminal
3. **Install Python packages**: Run the cell below

In [1]:
# Install required packages
# Note: These are installed as separate packages since LlamaIndex v0.10+
!pip install -q llama-index-core llama-index-llms-ollama llama-index-embeddings-huggingface
!pip install -q llama-index-readers-file pypdf sentence-transformers

## Step 1: Configure the LLM and Embedding Model

We'll use:
- **Ollama** with `gemma4` for text generation (runs on localhost:11434)
- **BGE-small** from HuggingFace for embeddings (384-dim, fast & accurate)

In [1]:
from llama_index.core import Settings
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Configure the LLM - Ollama runs locally on port 11434
Settings.llm = Ollama(
    model="gemma4",           # Use gemma4 (course default model)
    request_timeout=120.0,       # Timeout for generation
    temperature=0.1,             # Low temperature for factual responses
)

# Configure the embedding model - runs locally via sentence-transformers
Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5",  # 384-dim, ~130MB
)

print("LLM and Embedding model configured!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

LLM and Embedding model configured!


## Step 2: Load the PDF Document

LlamaIndex's `SimpleDirectoryReader` handles PDF parsing automatically.
Each page becomes a separate Document object with metadata.

In [2]:
from llama_index.core import SimpleDirectoryReader

# Load the PDF - using the attention paper as our example
documents = SimpleDirectoryReader(
    input_files=["./assets-resources/attention_paper.pdf"]
).load_data()

print(f"Loaded {len(documents)} pages from the PDF")
print(f"\nFirst page preview (first 500 chars):\n{documents[0].text[:500]}...")

Loaded 1 pages from the PDF

First page preview (first 500 chars):
%PDF-1.3
1 0 obj
<<
/Kids [ 4 0 R 5 0 R 6 0 R 7 0 R 8 0 R 9 0 R 10 0 R 11 0 R 12 0 R 13 0 R 14 0 R ]
/Type /Pages
/Count 11
>>
endobj
2 0 obj
<<
/Subject (Neural Information Processing Systems http\072\057\057nips\056cc\057)
/Publisher (Curran Associates\054 Inc\056)
/Language (en\055US)
/Created (2017)
/EventType (Poster)
/Description-Abstract (The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration\056 The...


## Step 3: Create the Vector Index

This step:
1. Chunks the documents (default: 1024 tokens with 20 overlap)
2. Generates embeddings for each chunk
3. Stores them in an in-memory vector store

For production, you'd use a persistent vector store like ChromaDB or FAISS.

In [3]:
from llama_index.core import VectorStoreIndex

# Create the index - this embeds all chunks
index = VectorStoreIndex.from_documents(
    documents,
    show_progress=True,  # Show embedding progress
)

print("\nIndex created successfully!") 

Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/365 [00:00<?, ?it/s]


Index created successfully!


## Step 4: Create the Query Engine

The query engine combines:
- **Retriever**: Finds relevant chunks using vector similarity
- **Response Synthesizer**: Generates answers using the LLM

`similarity_top_k=3` means we retrieve the 3 most relevant chunks.

In [4]:
# Create query engine with top-3 retrieval
query_engine = index.as_query_engine(
    similarity_top_k=3,  # Number of chunks to retrieve
)

print("Query engine ready!")

Query engine ready!


## Step 5: Query the Documents

Now we can ask questions about the paper! The RAG pipeline will:
1. Embed your question
2. Find similar chunks in the index
3. Send chunks + question to the LLM
4. Return the generated answer

In [ ]:
# Ask a question about the paper
response = query_engine.query("What is the main contribution of this paper?")

print("Answer:")
print(response.response)

Answer:
The provided information does not contain details regarding the main contribution of the paper.


In [6]:
# Let's try another question
response = query_engine.query("What is self-attention and how does it work?Answer in 5 bullet short points")

print("Answer:")
print(response.response)

Answer:
*   The architecture proposes a novel and simple network design that relies exclusively on an attention mechanism.
*   This approach allows the model to entirely dispense with the need for traditional recurrence and convolutions.
*   The attention mechanism is used to connect the encoder and the decoder within the model structure.
*   The model's performance on machine translation tasks is superior in quality compared to previous models.
*   Using this single model with 165 million parameters, the system demonstrated improved BLEU scores on both English-to-German and English-to-French translation tasks.


## Inspecting Retrieved Sources

One advantage of RAG is transparency - we can see which chunks were used to generate the answer.

In [7]:
# Inspect the source nodes (retrieved chunks)
print(f"Number of source chunks: {len(response.source_nodes)}\n")

for i, node in enumerate(response.source_nodes):
    print(f"--- Source {i+1} (score: {node.score:.3f}) ---")
    print(f"{node.text[:300]}...\n")

Number of source chunks: 3

--- Source 1 (score: 0.600) ---
%PDF-1.3
1 0 obj
<<
/Kids [ 4 0 R 5 0 R 6 0 R 7 0 R 8 0 R 9 0 R 10 0 R 11 0 R 12 0 R 13 0 R 14 0 R ]
/Type /Pages
/Count 11
>>
endobj
2 0 obj
<<
/Subject (Neural Information Processing Systems http\072\057\057nips\056cc\057)
/Publisher (Curran Associates\054 Inc\056)
/Language (en\055US)
/Created (2...

--- Source 2 (score: 0.539) ---
/AʏS<z;ud饋\g
t$VdȌwj0Y+2Q0q+啡e!fx*Va0k`U)M	PA҇\Ro?vG~'DBv_J~qkl
U$O\QϮ$qu
endstream
endobj
216 0 obj
<<
/Font <<
/F166 211 0 R
/F63 31 0 R
>>
/ProcSet [ /PDF /Text ]
>>
endobj
xref
0 217
0000000000 65535 f 
0000000009 00000 n 
0000000134 00000 n 
0000001979 00000 n 
00...

--- Source 3 (score: 0.538) ---
{o`{} rHRQRHʰĀg(s~
2B Jb[!tC[ں[6&LC)b(eH!
K4
(
S(+=դW+dɠP @!P(BRP( (LkJ2	(Bifr4(4f1
G@!!
iGgPHBa(dG>
 +4bq+4
B0h82mJ:2*($d@!
04Q2]08P(0(@"Z)I	BB8d4JBBXJ=@!...



## (Optional) Persist the Index

Save the index to disk so you don't need to re-embed documents each time.

In [8]:
# Save the index to disk
index.storage_context.persist(persist_dir="./storage/attention_paper")
print("Index saved to ./storage/attention_paper/")

Index saved to ./storage/attention_paper/


In [ ]:
# Uncomment to load:

# To load the index later:
# from llama_index.core import StorageContext, load_index_from_storage


# storage_context = StorageContext.from_defaults(persist_dir="./storage/attention_paper")
# loaded_index = load_index_from_storage(storage_context)
# query_engine = loaded_index.as_query_engine(similarity_top_k=3)

## Summary

You've built a complete local RAG pipeline! Key components:

| Component | Tool | Why |
|-----------|------|-----|
| LLM | Ollama (gemma4) | Local, no API keys, easy model management |
| Embeddings | HuggingFace (bge-small) | Fast, accurate, runs locally |
| Orchestration | LlamaIndex | Handles chunking, indexing, retrieval |

### Next Steps
- Try different models: `ollama pull mistral` or `ollama pull phi3`
- Use persistent storage: ChromaDB, FAISS
- Experiment with chunk sizes via `Settings.chunk_size`
- Add hybrid search (keyword + semantic)